In [1]:
!pip install google-smi
!google-smi

No TPU devices found.


In [5]:
import os

# Set the base directory for artifacts
use_google_drive = True
if use_google_drive:
  from google.colab import drive
  drive.mount('/content/drive')
  # Update the environment variable to point to Google Drive for future runs
  os.environ['NANOCHAT_BASE_DIR'] = '/content/drive/MyDrive/nanochat_data'
  print(f"Data persisted! NANOCHAT_BASE_DIR is now {os.environ['NANOCHAT_BASE_DIR']}")
else:
  os.environ['NANOCHAT_BASE_DIR'] = '/content/nanochat_data'
  print(f"NANOCHAT_BASE_DIR is now {os.environ['NANOCHAT_BASE_DIR']}")

Mounted at /content/drive
Data persisted! NANOCHAT_BASE_DIR is now /content/drive/MyDrive/nanochat_data


In [13]:
%cd /content/
!rm -rf nanochat
!git clone -b tpu https://github.com/megakilo/nanochat.git
%cd /content/nanochat

!mkdir -p $NANOCHAT_BASE_DIR

# Install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
# Add uv and cargo to the PATH
os.environ['PATH'] = f"/root/.cargo/bin:/root/.local/bin:{os.environ['PATH']}"

# Create a virtual environment and install dependencies
# !uv venv
!uv sync --extra tpu

# Install Rust/Cargo for the tokenizer
# !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
print('✨ Environment setup complete.')

/content
Cloning into 'nanochat'...
remote: Enumerating objects: 2123, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 2123 (delta 15), reused 12 (delta 9), pack-reused 2088 (from 3)
Receiving objects: 100% (2123/2123), 2.80 MiB | 7.40 MiB/s, done.
Resolving deltas: 100% (1312/1312), done.
/content/nanochat
downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 170 packages in 1ms
Installed 92 packages in 207ms                              
 + absl-py==2.5.1
 + annotated-types==0.8.0
 + anyio==4.15.1
 + certifi==2026.10.19
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cryptography==50.0.2
 + cwevals==0.0.1b1
 + distro==1.9.0
 + dnspython==2.9.0
 + email-validator==2.3.0
 + filelock==4.2.4
 + frozendict==2.4.7
 + fsspec==2026.9.0
 + googleapis-co

In [ ]:
!uv run python -m nanochat.dataset -n 35

In [ ]:
!uv run python -m scripts.tok_train --max-chars=2000000000
!uv run python -m scripts.tok_eval

In [ ]:
!uv run python -m scripts.base_train \
    --depth=24 \
    --model-tag=d24-tpu \
    --attn-impl=splash \
    --window-pattern=SSSL \
    --device-batch-size=8 \
    --eval-every=250 \
    --eval-tokens=5242880 \
    --core-metric-every=-1 \
    --core-metric-max-per-task=-1 \
    --sample-every=500 \
    --save-every=500

Uninstalled 1 package in 138ms
Installed 20 packages in 186ms                              
I0000 00:00:1792882123.565352   37641 pjrt_api.cc:119] GetPjrtApi was found for TPU at /content/nanochat/.venv/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1792882126.372244   37641 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1792882126.382026   37641 libtpu_version.cc:63] Setting libtpu version to: 0.0.51
/content/nanochat/.venv/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_

In [ ]:
!uv run python -m scripts.base_eval --device-batch-size=4

In [ ]:
!cat $NANOCHAT_BASE_DIR/base_eval/base_model_*.csv

In [ ]:
!uv run python -m scripts.chat_sft \
    --device-batch-size=4 \
    --eval-every=200 \
    --eval-tokens=524288 \
    --chatcore-every=999999